# GBM hyperparameter tuning

In [1]:
# Load data

import pandas as pd
import numpy as np 

train_df = pd.read_parquet("data/train_data.parquet")

train_df.index = pd.to_numeric(train_df.index, errors="coerce")

cols_to_drop = ["issue_d", "earliest_cr_line"]
train_df = train_df.drop(columns=cols_to_drop, errors="ignore")

In [2]:
# Separate features and targets

train_full_X = train_df.drop(["target", "target_annual_roi"], axis=1)
train_full_y_cat = train_df["target"]
train_full_y_reg = train_df["target_annual_roi"]

# Drop datetime features from the feature set

cols_to_drop = ["issue_d", "earliest_cr_line"]
train_full_X = train_full_X.drop(columns=cols_to_drop, errors="ignore")

In [3]:
# Create subsets of the data for different training sizes - chronological order is preserved, so we take the last N rows for each subset

train_1k_X = train_full_X.tail(1000)
train_1k_y_cat = train_full_y_cat.tail(1000)
train_1k_y_reg = train_full_y_reg.tail(1000)

train_10k_X = train_full_X.tail(10000)
train_10k_y_cat = train_full_y_cat.tail(10000)
train_10k_y_reg = train_full_y_reg.tail(10000)

train_100k_X = train_full_X.tail(100000)
train_100k_y_cat = train_full_y_cat.tail(100000)
train_100k_y_reg = train_full_y_reg.tail(100000)

In [4]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

cols_to_nominal_cat = train_df.select_dtypes(include=["object", "category"]).columns.tolist()

print("Categorical columns:")
for col in cols_to_nominal_cat:
    print(f"- {col}")

cardinality = train_df[cols_to_nominal_cat].nunique()
threshold_for_ohe = 5

cols_for_ohe = cardinality[cardinality <= threshold_for_ohe].index.tolist()
cols_for_te = cardinality[cardinality > threshold_for_ohe].index.tolist()

ohe_categories = []
for col in cols_for_ohe:
    unique_cats = train_df[col].dropna().unique().tolist()
    ohe_categories.append(unique_cats)

ohe_transformer = OneHotEncoder(
    categories=ohe_categories, 
    drop="if_binary", 
    handle_unknown="ignore", 
    sparse_output=False
)

target_transformer_nominal = TargetEncoder(target_type="binary", smooth="auto")

numeric_preprocessor = ColumnTransformer(
    transformers=[
        ("ohe", ohe_transformer, cols_for_ohe),
        ("target_enc", target_transformer_nominal, cols_for_te)
    ],
    remainder="passthrough", 
    verbose_feature_names_out=False
).set_output(transform="pandas")

imputed_numeric_preprocessor = make_pipeline(
    numeric_preprocessor,
    SimpleImputer(strategy="median")
).set_output(transform="pandas")

gbdt_preprocessor = "passthrough"

Categorical columns:
- home_ownership
- verification_status
- purpose
- addr_state
- initial_list_status
- application_type
- disbursement_method


## Classification

[Parameters](https://scikit-learn.org/1.6/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html)

### 1k

In [5]:
# Check the dates of 1k subset to ensure all data is from same month

print(train_1k_X["issue_d_month"].unique())
print(train_1k_X["issue_d_year"].unique())

<IntegerArray>
[10]
Length: 1, dtype: Int64
<IntegerArray>
[2016]
Length: 1, dtype: Int64


In [6]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Parameter tuning settings

timeout_seconds = 60*60*0.5
no_improvement_trials = 100

# Split the 1k dataset to evaluate on holdout after tuning
# Had problem with overfitting and extreme bad results on last 200 rows so switched the order

X_tuning = train_1k_X.tail(800)
y_tuning = train_1k_y_cat.tail(800)
X_val_holdout = train_1k_X.head(200)
y_val_holdout = train_1k_y_cat.head(200)

def objective_gbm(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth
        "subsample": trial.suggest_float("subsample", 0.3, 0.8), # Fraction of samples used for training each tree
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 50), # Min samples required to split a node
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 15, 100), # Min samples required in a leaf node
        "min_weight_fraction_leaf": trial.suggest_float("min_weight_fraction_leaf", 0, 0.5), # Min weight fraction required in a leaf node
        "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0, 1.0), # Min impurity decrease required to make a split
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 3, 31), # Max leaf nodes in the tree
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-5, 0.1, log=True), # Complexity parameter for pruning
        "random_state": 42, # Fixed random state for reproducibility
        "verbose": 0, # Set to 0 to disable verbose output during training
    }

    imputer_type = trial.suggest_categorical("imputer_type", ["simple", "knn", "iterative"])
    
    if imputer_type == "simple":
        strategy = trial.suggest_categorical("simple_strategy", ["mean", "median", "most_frequent"])
        imputer = SimpleImputer(strategy=strategy)
        
    elif imputer_type == "knn": # K Nearest Neighbors Imputer
        n_neighbors = trial.suggest_int("knn_neighbors", 3, 15)
        imputer = KNNImputer(n_neighbors=n_neighbors)
        
    else: # iterative
        imputer = IterativeImputer(max_iter=10, random_state=42)

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        imputer,
        GradientBoostingClassifier(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_gbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_gbm.optimize(objective_gbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_gbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_gbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_gbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_gbm)
fig1.show()
    
fig2 = plot_param_importances(study_gbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_gbm,
    params=[
        "n_estimators", "learning_rate", "max_depth", "subsample", 
        "min_samples_split", "min_samples_leaf", "min_weight_fraction_leaf",
        "min_impurity_decrease", "max_features", "max_leaf_nodes",
        "ccp_alpha", "imputer_type"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_1k_history.html")
fig2.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_1k_importance.html")
fig3.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_1k_parallel.html")


[I 2026-04-26 17:46:04,596] A new study created in memory with name: no-name-c9a82b73-14b6-4473-a548-100278ae53a1
[I 2026-04-26 17:46:06,479] Trial 1 finished with value: 0.5 and parameters: {'n_estimators': 108, 'learning_rate': 0.09062924929980061, 'max_depth': 3, 'subsample': 0.3192861676017728, 'min_samples_split': 9, 'min_samples_leaf': 75, 'min_weight_fraction_leaf': 0.3859291324222454, 'min_impurity_decrease': 0.0036529603434893865, 'max_features': 'sqrt', 'max_leaf_nodes': 29, 'ccp_alpha': 0.07026368966353382, 'imputer_type': 'simple', 'simple_strategy': 'median'}. Best is trial 1 with value: 0.5.
[I 2026-04-26 17:46:10,790] Trial 3 finished with value: 0.6749297828608173 and parameters: {'n_estimators': 428, 'learning_rate': 0.07501730702211001, 'max_depth': 3, 'subsample': 0.7455351167629212, 'min_samples_split': 30, 'min_samples_leaf': 44, 'min_weight_fraction_leaf': 0.18860307869930815, 'min_impurity_decrease': 0.870670870175372, 'max_features': 'log2', 'max_leaf_nodes': 19


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 17:56:42,982] Trial 164 finished with value: 0.6753577648405235 and parameters: {'n_estimators': 966, 'learning_rate': 0.13896015934613995, 'max_depth': 3, 'subsample': 0.7193004481259445, 'min_samples_split': 40, 'min_samples_leaf': 36, 'min_weight_fraction_leaf': 0.13153549637821382, 'min_impurity_decrease': 0.3199890692796177, 'max_features': 'log2', 'max_leaf_nodes': 18, 'ccp_alpha': 0.00010381352081574507, 'imputer_type': 'simple', 'simple_strategy': 'median'}. Best is trial 63 with value: 0.6879237142168177.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 17:56:44,111] Trial 165 finished with value: 0.6685393533669396 and parameters: {'n_estimators': 968, 'learning_rate': 0.13794600730226408, 'max_depth': 3, 'subsample': 0.7189331774352034, 'min_samples_split': 50, 'min_samples_leaf': 42, 'min_weight_fraction_leaf': 0.027118704347435563, 'min_impurity_decrease': 0.3214769195666886, 'max_features': 'log2', 'max_leaf_nodes': 23, 'ccp_alpha': 0.00011335357356109006, 'imputer_type': 'simple', 'simple_strategy': 'median'}. Best is trial 63 with value: 0.6879237142168177.
[I 2026-04-26 17:56:44,153] Trial 166 finished with value: 0.6879020213502972 and parameters: {'n_estimators': 972, 'learning_rate': 0.13694155600150612, 'max_depth': 3, 'subsample': 0.7180550060684775, 'min_samples_split': 39, 'min_samples_leaf': 36, 'min_weight_fraction_leaf': 0.026162867424174206, 'min_impurity_decrease': 0.31813411156285404, 'max_features': 'log2', 'max_leaf_nodes': 22, 'ccp_alpha': 7.164564945859198e-05, 'imputer_type': 'simple', 'simple_s


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 17:56:44,591] Trial 168 finished with value: 0.6659994985857055 and parameters: {'n_estimators': 579, 'learning_rate': 0.039108648181999967, 'max_depth': 3, 'subsample': 0.7448439410200682, 'min_samples_split': 42, 'min_samples_leaf': 43, 'min_weight_fraction_leaf': 0.08743363524664968, 'min_impurity_decrease': 0.5121286959039654, 'max_features': 'log2', 'max_leaf_nodes': 23, 'ccp_alpha': 4.584963022634636e-05, 'imputer_type': 'simple', 'simple_strategy': 'median'}. Best is trial 63 with value: 0.6879237142168177.
[I 2026-04-26 17:56:44,611] Trial 167 finished with value: 0.6714504556745936 and parameters: {'n_estimators': 964, 'learning_rate': 0.13974881845825127, 'max_depth': 3, 'subsample': 0.664427932193141, 'min_samples_split': 50, 'min_samples_leaf': 36, 'min_weight_fraction_leaf': 0.15163579540885175, 'min_impurity_decrease': 0.5161277731475699, 'max_features': 'log2', 'max_leaf_nodes': 23, 'ccp_alpha': 2.0239078629530418e-05, 'imputer_type': 'simple', 'simple_stra


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 17:56:44,911] Trial 170 finished with value: 0.6610286935286935 and parameters: {'n_estimators': 929, 'learning_rate': 0.14152725209986602, 'max_depth': 3, 'subsample': 0.661954522107867, 'min_samples_split': 47, 'min_samples_leaf': 36, 'min_weight_fraction_leaf': 0.0837362145603805, 'min_impurity_decrease': 0.32497227470423296, 'max_features': 'log2', 'max_leaf_nodes': 21, 'ccp_alpha': 0.00010990730384978061, 'imputer_type': 'simple', 'simple_strategy': 'median'}. Best is trial 63 with value: 0.6879237142168177.



[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST AUC: 0.6879
BEST PARAMETERS:
best_params = {
    "n_estimators": 579,
    "learning_rate": 0.11372034355457401,
    "max_depth": 3,
    "subsample": 0.6608071029790215,
    "min_samples_split": 44,
    "min_samples_leaf": 36,
    "min_weight_fraction_leaf": 0.13651164177932174,
    "min_impurity_decrease": 0.91953008742171,
    "max_features": "log2",
    "max_leaf_nodes": 22,
    "ccp_alpha": 2.1238742124806448e-05,
    "imputer_type": "simple",
    "simple_strategy": "mean",
}

--- PARAMETER IMPORTANCE ---
  ccp_alpha           : 0.8227
  min_samples_leaf    : 0.0571
  min_weight_fraction_leaf: 0.0450
  min_impurity_decrease: 0.0198
  subsample           : 0.0183
  max_features        : 0.0142
  max_depth           : 0.0104
  min_samples_split   : 0.0074
  learning_rate       : 0.0022
  n_estimators        : 0.0018
  max_leaf_nodes      : 0.0009
  imputer_type        : 0.0002


In [7]:
# Check overfit on holdout set with best parameters from tuning

best_params = study_gbm.best_params.copy()
best_params["random_state"] = 42
best_params["verbose"] = 0

imputer_type = best_params.pop("imputer_type")

if imputer_type == "simple":
    strategy = best_params.pop("simple_strategy")
    best_imputer = SimpleImputer(strategy=strategy)
    print(f"SimpleImputer strategy: {strategy}")
elif imputer_type == "knn":
    n_neighbors = best_params.pop("knn_neighbors")
    best_imputer = KNNImputer(n_neighbors=n_neighbors)
    print(f"KNNImputer n_neighbors: {n_neighbors}")
else:
    best_imputer = IterativeImputer(max_iter=10, random_state=42)
    print("IterativeImputer selected")

best_params.pop("simple_strategy", None)
best_params.pop("knn_neighbors", None)
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    best_imputer,
    GradientBoostingClassifier(**best_params)
)

final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print(f"Optuna Val AUC: {study_gbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")

SimpleImputer strategy: mean
BEST PARAMS: {'n_estimators': 579, 'learning_rate': 0.11372034355457401, 'max_depth': 3, 'subsample': 0.6608071029790215, 'min_samples_split': 44, 'min_samples_leaf': 36, 'min_weight_fraction_leaf': 0.13651164177932174, 'min_impurity_decrease': 0.91953008742171, 'max_features': 'log2', 'max_leaf_nodes': 22, 'ccp_alpha': 2.1238742124806448e-05, 'random_state': 42, 'verbose': 0}
Optuna Val AUC: 0.6879
Holdout Test AUC: 0.6716


### 10k

In [8]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Parameter tuning settings

timeout_seconds = 60*60*1
no_improvement_trials = 100

# Split the 10k dataset to evaluate on holdout after tuning

X_tuning = train_10k_X.head(8000)
y_tuning = train_10k_y_cat.head(8000)
X_val_holdout = train_10k_X.tail(2000)
y_val_holdout = train_10k_y_cat.tail(2000)

def objective_gbm(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth
        "subsample": trial.suggest_float("subsample", 0.3, 0.8), # Fraction of samples used for training each tree
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 50), # Min samples required to split a node
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 15, 100), # Min samples required in a leaf node
        "min_weight_fraction_leaf": trial.suggest_float("min_weight_fraction_leaf", 0, 0.5), # Min weight fraction required in a leaf node
        "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0, 1.0), # Min impurity decrease required to make a split
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 3, 31), # Max leaf nodes in the tree
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-5, 0.1, log=True), # Complexity parameter for pruning
        "random_state": 42, # Fixed random state for reproducibility
        "verbose": 0, # Set to 0 to disable verbose output during training
    }

    imputer_type = trial.suggest_categorical("imputer_type", ["simple", "knn", "iterative"])
    
    if imputer_type == "simple":
        strategy = trial.suggest_categorical("simple_strategy", ["mean", "median", "most_frequent"])
        imputer = SimpleImputer(strategy=strategy)
        
    elif imputer_type == "knn": # K Nearest Neighbors Imputer
        n_neighbors = trial.suggest_int("knn_neighbors", 3, 15)
        imputer = KNNImputer(n_neighbors=n_neighbors)
        
    else: # iterative
        imputer = IterativeImputer(max_iter=10, random_state=42)

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        imputer,
        GradientBoostingClassifier(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_gbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_gbm.optimize(objective_gbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_gbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_gbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_gbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_gbm)
fig1.show()
    
fig2 = plot_param_importances(study_gbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_gbm,
    params=[
        "n_estimators", "learning_rate", "max_depth", "subsample", 
        "min_samples_split", "min_samples_leaf", "min_weight_fraction_leaf",
        "min_impurity_decrease", "max_features", "max_leaf_nodes",
        "ccp_alpha", "imputer_type"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_10k_history.html")
fig2.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_10k_importance.html")
fig3.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_10k_parallel.html")


[I 2026-04-26 18:00:47,455] A new study created in memory with name: no-name-15372b1f-5433-4373-bd11-8a8c79001c8a
[I 2026-04-26 18:01:05,259] Trial 6 finished with value: 0.6803249621273754 and parameters: {'n_estimators': 140, 'learning_rate': 0.045304489386671906, 'max_depth': 7, 'subsample': 0.45176123146085156, 'min_samples_split': 16, 'min_samples_leaf': 28, 'min_weight_fraction_leaf': 0.41101940679493615, 'min_impurity_decrease': 0.8926857182273078, 'max_features': None, 'max_leaf_nodes': 23, 'ccp_alpha': 0.0005404855245966782, 'imputer_type': 'simple', 'simple_strategy': 'median'}. Best is trial 6 with value: 0.6803249621273754.
[I 2026-04-26 18:01:06,499] Trial 2 finished with value: 0.6801115123095509 and parameters: {'n_estimators': 451, 'learning_rate': 0.024214335237315435, 'max_depth': 2, 'subsample': 0.765126877697843, 'min_samples_split': 9, 'min_samples_leaf': 52, 'min_weight_fraction_leaf': 0.4011899576152462, 'min_impurity_decrease': 0.967749307228406, 'max_features':


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 18:56:29,961] Trial 220 finished with value: 0.6850404435218579 and parameters: {'n_estimators': 653, 'learning_rate': 0.01799960504987084, 'max_depth': 3, 'subsample': 0.6261679076872542, 'min_samples_split': 30, 'min_samples_leaf': 70, 'min_weight_fraction_leaf': 0.3695054126326477, 'min_impurity_decrease': 0.8222102554685523, 'max_features': 'log2', 'max_leaf_nodes': 15, 'ccp_alpha': 3.113070743421965e-05, 'imputer_type': 'simple', 'simple_strategy': 'median'}. Best is trial 117 with value: 0.70266862638139.
[I 2026-04-26 18:56:29,996] Trial 219 finished with value: 0.7004533029366256 and parameters: {'n_estimators': 653, 'learning_rate': 0.02550000188657993, 'max_depth': 3, 'subsample': 0.6278379924474424, 'min_samples_split': 30, 'min_samples_leaf': 71, 'min_weight_fraction_leaf': 0.023818883956792103, 'min_impurity_decrease': 0.8261123230602466, 'max_features': 'log2', 'max_leaf_nodes': 14, 'ccp_alpha': 3.218758580488221e-05, 'imputer_type': 'simple', 'simple_strate


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 18:56:31,136] Trial 218 finished with value: 0.699910896642207 and parameters: {'n_estimators': 696, 'learning_rate': 0.01776112712073787, 'max_depth': 3, 'subsample': 0.6283065725709492, 'min_samples_split': 30, 'min_samples_leaf': 70, 'min_weight_fraction_leaf': 0.02323163320304042, 'min_impurity_decrease': 0.8249447472466235, 'max_features': 'log2', 'max_leaf_nodes': 15, 'ccp_alpha': 3.095050427617139e-05, 'imputer_type': 'simple', 'simple_strategy': 'median'}. Best is trial 117 with value: 0.70266862638139.


...


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 18:56:37,278] Trial 224 finished with value: 0.7016301336909906 and parameters: {'n_estimators': 692, 'learning_rate': 0.018113462966794454, 'max_depth': 3, 'subsample': 0.630801233805004, 'min_samples_split': 30, 'min_samples_leaf': 70, 'min_weight_fraction_leaf': 0.010031251312168395, 'min_impurity_decrease': 0.7794000456957952, 'max_features': 'log2', 'max_leaf_nodes': 14, 'ccp_alpha': 3.690179868823148e-05, 'imputer_type': 'simple', 'simple_strategy': 'median'}. Best is trial 117 with value: 0.70266862638139.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-26 18:56:37,783] Trial 223 finished with value: 0.7012826509608578 and parameters: {'n_estimators': 689, 'learning_rate': 0.017556912427122165, 'max_depth': 3, 'subsample': 0.6452505077654592, 'min_samples_split': 29, 'min_samples_leaf': 70, 'min_weight_fraction_leaf': 0.007117622328158717, 'min_impurity_decrease': 0.5272572785627669, 'max_features': 'log2', 'max_leaf_nodes': 14, 'ccp_alpha': 3.847463969627054e-05, 'imputer_type': 'simple', 'simple_strategy': 'median'}. Best is trial 117 with value: 0.70266862638139.



[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST AUC: 0.7027
BEST PARAMETERS:
best_params = {
    "n_estimators": 794,
    "learning_rate": 0.01098780117093146,
    "max_depth": 3,
    "subsample": 0.634594524243599,
    "min_samples_split": 27,
    "min_samples_leaf": 79,
    "min_weight_fraction_leaf": 0.002500791243609685,
    "min_impurity_decrease": 0.5395017070575661,
    "max_features": "log2",
    "max_leaf_nodes": 8,
    "ccp_alpha": 2.565584578912088e-05,
    "imputer_type": "simple",
    "simple_strategy": "median",
}

--- PARAMETER IMPORTANCE ---
  ccp_alpha           : 0.9965
  max_leaf_nodes      : 0.0014
  min_samples_split   : 0.0005
  subsample           : 0.0005
  n_estimators        : 0.0005
  min_weight_fraction_leaf: 0.0004
  imputer_type        : 0.0002
  max_features        : 0.0000
  min_impurity_decrease: 0.0000
  learning_rate       : 0.0000
  max_depth           : 0.0000
  min_samples_leaf    : 0.0000


In [9]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_gbm.best_params.copy()

# Extract imputer settings safely before applying scaling trick
imputer_type = best_params.pop("imputer_type")

if imputer_type == "simple":
    strategy = best_params.pop("simple_strategy")
    best_imputer = SimpleImputer(strategy=strategy)
    print(f"SimpleImputer strategy: {strategy}")
elif imputer_type == "knn":
    n_neighbors = best_params.pop("knn_neighbors")
    best_imputer = KNNImputer(n_neighbors=n_neighbors)
    print(f"KNNImputer n_neighbors: {n_neighbors}")
else:
    best_imputer = IterativeImputer(max_iter=10, random_state=42)
    print("IterativeImputer selected")

# Clean up parameter dictionary
best_params.pop("simple_strategy", None)
best_params.pop("knn_neighbors", None)

# Scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

best_params["n_estimators"] = original_trees * 2
best_params["learning_rate"] = original_lr / 2

print(f"\n[Scaling Trick Applied] SKLEARN GBM: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    best_imputer,
    GradientBoostingClassifier(**best_params)
)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Val AUC:   {study_gbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")
print("="*40)

SimpleImputer strategy: median

[Scaling Trick Applied] SKLEARN GBM: Trees 794 -> 1588, LR 0.0110 -> 0.0055
BEST PARAMS: {'n_estimators': 1588, 'learning_rate': 0.00549390058546573, 'max_depth': 3, 'subsample': 0.634594524243599, 'min_samples_split': 27, 'min_samples_leaf': 79, 'min_weight_fraction_leaf': 0.002500791243609685, 'min_impurity_decrease': 0.5395017070575661, 'max_features': 'log2', 'max_leaf_nodes': 8, 'ccp_alpha': 2.565584578912088e-05, 'random_state': 42, 'verbose': 0}

Optuna Val AUC:   0.7027
Holdout Test AUC: 0.6981


### 100k

In [5]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Parameter tuning settings

timeout_seconds = 60*60*8
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning

X_tuning = train_100k_X.head(80000)
y_tuning = train_100k_y_cat.head(80000)
X_val_holdout = train_100k_X.tail(20000)
y_val_holdout = train_100k_y_cat.tail(20000)

def objective_gbm(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 20), # Max tree depth
        "subsample": trial.suggest_float("subsample", 0.1, 1.0), # Fraction of samples used for training each tree
        "min_samples_split": trial.suggest_int("min_samples_split", 10, 500), # Min samples required to split a node
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 500), # Min samples required in a leaf node
        "min_weight_fraction_leaf": trial.suggest_float("min_weight_fraction_leaf", 0, 0.5), # Min weight fraction required in a leaf node
        "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0, 1.0), # Min impurity decrease required to make a split
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 5, 512), # Max leaf nodes in the tree
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-5, 0.1, log=True), # Complexity parameter for pruning
        "random_state": 42, # Fixed random state for reproducibility
        "verbose": 0, # Set to 0 to disable verbose output during training
    }

    imputer_type = trial.suggest_categorical("imputer_type", ["simple", "knn", "iterative"])
    
    if imputer_type == "simple":
        strategy = trial.suggest_categorical("simple_strategy", ["mean", "median", "most_frequent"])
        imputer = SimpleImputer(strategy=strategy)
        
    elif imputer_type == "knn": # K Nearest Neighbors Imputer
        n_neighbors = trial.suggest_int("knn_neighbors", 3, 50)
        imputer = KNNImputer(n_neighbors=n_neighbors)
        
    else: # iterative
        imputer = IterativeImputer(max_iter=10, random_state=42)

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        imputer,
        GradientBoostingClassifier(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_gbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_gbm.optimize(objective_gbm, timeout=timeout_seconds, n_jobs=1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_gbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_gbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_gbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_gbm)
fig1.show()
    
fig2 = plot_param_importances(study_gbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_gbm,
    params=[
        "n_estimators", "learning_rate", "max_depth", "subsample", 
        "min_samples_split", "min_samples_leaf", "min_weight_fraction_leaf",
        "min_impurity_decrease", "max_features", "max_leaf_nodes",
        "ccp_alpha", "imputer_type"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_100k_history.html")
fig2.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_100k_importance.html")
fig3.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_100k_parallel.html")


[I 2026-04-28 15:46:41,438] A new study created in memory with name: no-name-7877ee6a-15e6-4adc-90dd-c870eab75826
[I 2026-04-28 15:47:36,330] Trial 0 finished with value: 0.5 and parameters: {'n_estimators': 437, 'learning_rate': 0.1667521176194013, 'max_depth': 15, 'subsample': 0.6387926357773329, 'min_samples_split': 86, 'min_samples_leaf': 86, 'min_weight_fraction_leaf': 0.02904180608409973, 'min_impurity_decrease': 0.8661761457749352, 'max_features': 'log2', 'max_leaf_nodes': 497, 'ccp_alpha': 0.021368329072358756, 'imputer_type': 'simple', 'simple_strategy': 'median'}. Best is trial 0 with value: 0.5.
[I 2026-04-28 15:52:57,152] Trial 1 finished with value: 0.6840495940636035 and parameters: {'n_estimators': 362, 'learning_rate': 0.04777437867054351, 'max_depth': 4, 'subsample': 0.3629301836816964, 'min_samples_split': 189, 'min_samples_leaf': 233, 'min_weight_fraction_leaf': 0.3925879806965068, 'min_impurity_decrease': 0.19967378215835974, 'max_features': 'log2', 'max_leaf_nodes'


BEST AUC: 0.7077
BEST PARAMETERS:
best_params = {
    "n_estimators": 475,
    "learning_rate": 0.1648269774352038,
    "max_depth": 16,
    "subsample": 0.9090892401385037,
    "min_samples_split": 443,
    "min_samples_leaf": 75,
    "min_weight_fraction_leaf": 0.09091249242263949,
    "min_impurity_decrease": 0.6068460342082242,
    "max_features": None,
    "max_leaf_nodes": 82,
    "ccp_alpha": 1.9782964485238465e-05,
    "imputer_type": "simple",
    "simple_strategy": "mean",
}

--- PARAMETER IMPORTANCE ---
  ccp_alpha           : 0.8912
  n_estimators        : 0.0499
  min_samples_split   : 0.0283
  learning_rate       : 0.0156
  max_leaf_nodes      : 0.0138
  min_weight_fraction_leaf: 0.0004
  min_samples_leaf    : 0.0003
  max_depth           : 0.0002
  subsample           : 0.0001
  imputer_type        : 0.0000
  max_features        : 0.0000
  min_impurity_decrease: 0.0000


In [6]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_gbm.best_params.copy()

# Extract imputer settings safely before applying scaling trick
imputer_type = best_params.pop("imputer_type")

if imputer_type == "simple":
    strategy = best_params.pop("simple_strategy")
    best_imputer = SimpleImputer(strategy=strategy)
    print(f"SimpleImputer strategy: {strategy}")
elif imputer_type == "knn":
    n_neighbors = best_params.pop("knn_neighbors")
    best_imputer = KNNImputer(n_neighbors=n_neighbors)
    print(f"KNNImputer n_neighbors: {n_neighbors}")
else:
    best_imputer = IterativeImputer(max_iter=10, random_state=42)
    print("IterativeImputer selected")

# Clean up parameter dictionary
best_params.pop("simple_strategy", None)
best_params.pop("knn_neighbors", None)

# Scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

best_params["n_estimators"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10

print(f"\n[Scaling Trick Applied] SKLEARN GBM: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    best_imputer,
    GradientBoostingClassifier(**best_params)
)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Val AUC:   {study_gbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")
print("="*40)

SimpleImputer strategy: mean

[Scaling Trick Applied] SKLEARN GBM: Trees 475 -> 4750, LR 0.1648 -> 0.0165
BEST PARAMS: {'n_estimators': 4750, 'learning_rate': 0.01648269774352038, 'max_depth': 16, 'subsample': 0.9090892401385037, 'min_samples_split': 443, 'min_samples_leaf': 75, 'min_weight_fraction_leaf': 0.09091249242263949, 'min_impurity_decrease': 0.6068460342082242, 'max_features': None, 'max_leaf_nodes': 82, 'ccp_alpha': 1.9782964485238465e-05, 'random_state': 42, 'verbose': 0}

Optuna Val AUC:   0.7077
Holdout Test AUC: 0.7098


### Whole training data set

In [ ]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Parameter tuning settings

timeout_seconds = 60*60*4
no_improvement_trials = 100

# Split the dataset to evaluate on holdout after tuning

split_index = int(len(train_full_X) * 0.8)

X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_cat[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_cat[split_index:]

def objective_gbm(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 20), # Max tree depth
        "subsample": trial.suggest_float("subsample", 0.1, 1.0), # Fraction of samples used for training each tree
        "min_samples_split": trial.suggest_int("min_samples_split", 10, 500), # Min samples required to split a node
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 500), # Min samples required in a leaf node
        "min_weight_fraction_leaf": trial.suggest_float("min_weight_fraction_leaf", 0, 0.5), # Min weight fraction required in a leaf node
        "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0, 1.0), # Min impurity decrease required to make a split
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 5, 512), # Max leaf nodes in the tree
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-5, 0.1, log=True), # Complexity parameter for pruning
        "random_state": 42, # Fixed random state for reproducibility
        "verbose": 0, # Set to 0 to disable verbose output during training
    }

    imputer_type = trial.suggest_categorical("imputer_type", ["simple", "knn", "iterative"])
    
    if imputer_type == "simple":
        strategy = trial.suggest_categorical("simple_strategy", ["mean", "median", "most_frequent"])
        imputer = SimpleImputer(strategy=strategy)
        
    elif imputer_type == "knn": # K Nearest Neighbors Imputer
        n_neighbors = trial.suggest_int("knn_neighbors", 3, 50)
        imputer = KNNImputer(n_neighbors=n_neighbors)
        
    else: # iterative
        imputer = IterativeImputer(max_iter=10, random_state=42)

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        imputer,
        GradientBoostingClassifier(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_gbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_gbm.optimize(objective_gbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_gbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_gbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_gbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_gbm)
fig1.show()
    
fig2 = plot_param_importances(study_gbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_gbm,
    params=[
        "n_estimators", "learning_rate", "max_depth", "subsample", 
        "min_samples_split", "min_samples_leaf", "min_weight_fraction_leaf",
        "min_impurity_decrease", "max_features", "max_leaf_nodes",
        "ccp_alpha", "imputer_type"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_full_history.html")
fig2.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_full_importance.html")
fig3.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_full_parallel.html")


[I 2026-04-22 00:55:36,440] A new study created in memory with name: no-name-e6e86f3f-4af6-4c6c-bb30-bebed2ab97f3
[I 2026-04-22 00:56:27,391] Trial 3 finished with value: 0.7509030339384353 and parameters: {'boosting_type': 'goss', 'num_leaves': 377, 'max_depth': 2, 'learning_rate': 0.1903428179678173, 'scale_pos_weight': 6.082125486473243, 'min_split_gain': 26.39142477642419, 'min_child_weight': 4.8185524092618045e-05, 'min_child_samples': 290, 'colsample_bytree': 0.5066837006445346, 'reg_alpha': 0.008590451603378712, 'reg_lambda': 0.0006321751034820287, 'colsample_bynode': 0.6989172440719156, 'min_data_per_group': 976, 'max_cat_threshold': 632, 'cat_l2': 4.676533112163103e-06, 'cat_smooth': 34.70861426629406, 'max_cat_to_onehot': 37, 'max_bin': 433, 'top_rate': 0.7237115739628408, 'other_rate': 0.1719406111579936, 'n_estimators': 6050}. Best is trial 3 with value: 0.7509030339384353.
[I 2026-04-22 00:56:59,196] Trial 6 finished with value: 0.7442250170611076 and parameters: {'boostin


BEST AUC: 0.7566
BEST PARAMETERS:
best_params = {
    "boosting_type": "goss",
    "num_leaves": 298,
    "max_depth": 20,
    "learning_rate": 0.002488325776616227,
    "scale_pos_weight": 5.888271675991455,
    "min_split_gain": 1.6197064284930547,
    "min_child_weight": 0.0002644551677947709,
    "min_child_samples": 371,
    "colsample_bytree": 0.5316730321670442,
    "reg_alpha": 0.000426119399156468,
    "reg_lambda": 10.280323813334233,
    "colsample_bynode": 0.5281296236914715,
    "min_data_per_group": 996,
    "max_cat_threshold": 678,
    "cat_l2": 1.640465366517915e-08,
    "cat_smooth": 22.561055503199018,
    "max_cat_to_onehot": 45,
    "max_bin": 454,
    "top_rate": 0.18095067523535055,
    "other_rate": 0.31987875938170135,
    "n_estimators": 6725,
}

--- PARAMETER IMPORTANCE ---
  max_depth           : 0.2899
  min_data_per_group  : 0.1874
  max_bin             : 0.1858
  boosting_type       : 0.0868
  learning_rate       : 0.0573
  min_child_samples   : 0.0480
 

In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_gbm.best_params.copy()

# Extract imputer settings safely before applying scaling trick
imputer_type = best_params.pop("imputer_type")

if imputer_type == "simple":
    strategy = best_params.pop("simple_strategy")
    best_imputer = SimpleImputer(strategy=strategy)
    print(f"SimpleImputer strategy: {strategy}")
elif imputer_type == "knn":
    n_neighbors = best_params.pop("knn_neighbors")
    best_imputer = KNNImputer(n_neighbors=n_neighbors)
    print(f"KNNImputer n_neighbors: {n_neighbors}")
else:
    best_imputer = IterativeImputer(max_iter=10, random_state=42)
    print("IterativeImputer selected")

# Clean up parameter dictionary
best_params.pop("simple_strategy", None)
best_params.pop("knn_neighbors", None)

# Scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

best_params["n_estimators"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10

print(f"\n[Scaling Trick Applied] SKLEARN GBM: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    best_imputer,
    GradientBoostingClassifier(**best_params)
)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Val AUC:   {study_gbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")
print("="*40)

Optuna Val AUC: 0.7566
Holdout Test AUC: 0.7298


## Regression

[Parameters](https://scikit-learn.org/1.6/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html)

### 1k

In [4]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

cols_to_nominal_cat = train_df.select_dtypes(include=["object", "category"]).columns.tolist()

print("Categorical columns:")
for col in cols_to_nominal_cat:
    print(f"- {col}")

cardinality = train_df[cols_to_nominal_cat].nunique()
threshold_for_ohe = 5

cols_for_ohe = cardinality[cardinality <= threshold_for_ohe].index.tolist()
cols_for_te = cardinality[cardinality > threshold_for_ohe].index.tolist()

ohe_categories = []
for col in cols_for_ohe:
    unique_cats = train_df[col].dropna().unique().tolist()
    ohe_categories.append(unique_cats)

ohe_transformer = OneHotEncoder(
    categories=ohe_categories, 
    drop="if_binary", 
    handle_unknown="ignore", 
    sparse_output=False
)

target_transformer_nominal = TargetEncoder(target_type="continuous", smooth="auto")

numeric_preprocessor = ColumnTransformer(
    transformers=[
        ("ohe", ohe_transformer, cols_for_ohe),
        ("target_enc", target_transformer_nominal, cols_for_te)
    ],
    remainder="passthrough", 
    verbose_feature_names_out=False
).set_output(transform="pandas")

imputed_numeric_preprocessor = make_pipeline(
    numeric_preprocessor,
    SimpleImputer(strategy="median")
).set_output(transform="pandas")

gbdt_preprocessor = "passthrough"

Categorical columns:
- home_ownership
- verification_status
- purpose
- addr_state
- initial_list_status
- application_type
- disbursement_method


In [6]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Parameter tuning settings

timeout_seconds = 60*60*0.5
no_improvement_trials = 100

# Split the 1k dataset to evaluate on holdout after tuning
# Had problem with overfitting and extreme bad results on last 200 rows so switched the order

X_tuning = train_1k_X.tail(800)
y_tuning = train_1k_y_reg.tail(800)
X_val_holdout = train_1k_X.head(200)
y_val_holdout = train_1k_y_reg.head(200)

def objective_gbm(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth
        "subsample": trial.suggest_float("subsample", 0.3, 0.8), # Fraction of samples used for training each tree
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 50), # Min samples required to split a node
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 15, 100), # Min samples required in a leaf node
        "min_weight_fraction_leaf": trial.suggest_float("min_weight_fraction_leaf", 0, 0.5), # Min weight fraction required in a leaf node
        "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0, 1.0), # Min impurity decrease required to make a split
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 3, 31), # Max leaf nodes in the tree
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-5, 0.1, log=True), # Complexity parameter for pruning
        "random_state": 42, # Fixed random state for reproducibility
        "verbose": 0, # Set to 0 to disable verbose output during training
    }

    imputer_type = trial.suggest_categorical("imputer_type", ["simple", "knn", "iterative"])
    
    if imputer_type == "simple":
        strategy = trial.suggest_categorical("simple_strategy", ["mean", "median", "most_frequent"])
        imputer = SimpleImputer(strategy=strategy)
        
    elif imputer_type == "knn": # K Nearest Neighbors Imputer
        n_neighbors = trial.suggest_int("knn_neighbors", 3, 15)
        imputer = KNNImputer(n_neighbors=n_neighbors)
        
    else: # iterative
        imputer = IterativeImputer(max_iter=10, random_state=42)

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        imputer,
        GradientBoostingRegressor(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_gbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_gbm.optimize(objective_gbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_gbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_gbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_gbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_gbm)
fig1.show()
    
fig2 = plot_param_importances(study_gbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_gbm,
    params=[
        "n_estimators", "learning_rate", "max_depth", "subsample", 
        "min_samples_split", "min_samples_leaf", "min_weight_fraction_leaf",
        "min_impurity_decrease", "max_features", "max_leaf_nodes",
        "ccp_alpha", "imputer_type"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_1k_history.html")
fig2.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_1k_importance.html")
fig3.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_1k_parallel.html")


[I 2026-04-27 18:49:07,364] A new study created in memory with name: no-name-4e2fe14f-399e-4fdf-a75e-242aa9dd358b
[I 2026-04-27 18:49:11,617] Trial 4 finished with value: 0.349719690780588 and parameters: {'n_estimators': 332, 'learning_rate': 0.15607544589389402, 'max_depth': 8, 'subsample': 0.5917742666572441, 'min_samples_split': 20, 'min_samples_leaf': 78, 'min_weight_fraction_leaf': 0.4883777725622213, 'min_impurity_decrease': 0.06557678626775854, 'max_features': 'log2', 'max_leaf_nodes': 31, 'ccp_alpha': 1.159936616802312e-05, 'imputer_type': 'simple', 'simple_strategy': 'median'}. Best is trial 4 with value: 0.349719690780588.
[I 2026-04-27 18:49:13,322] Trial 3 finished with value: 0.3487407154249822 and parameters: {'n_estimators': 309, 'learning_rate': 0.01541835109446497, 'max_depth': 8, 'subsample': 0.30456744005727177, 'min_samples_split': 39, 'min_samples_leaf': 45, 'min_weight_fraction_leaf': 0.008865765741691567, 'min_impurity_decrease': 0.5027559210961271, 'max_feature


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-27 19:07:35,205] Trial 520 finished with value: 0.3505631436082363 and parameters: {'n_estimators': 975, 'learning_rate': 0.013434373569291022, 'max_depth': 8, 'subsample': 0.7204887897242763, 'min_samples_split': 38, 'min_samples_leaf': 23, 'min_weight_fraction_leaf': 0.39662418137555305, 'min_impurity_decrease': 0.7521317303977277, 'max_features': 'log2', 'max_leaf_nodes': 6, 'ccp_alpha': 2.6376770139584957e-05, 'imputer_type': 'simple', 'simple_strategy': 'mean'}. Best is trial 419 with value: 0.3453101317437835.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-27 19:07:35,539] Trial 521 finished with value: 0.3466356322507563 and parameters: {'n_estimators': 973, 'learning_rate': 0.013591150455681135, 'max_depth': 8, 'subsample': 0.30021102796414306, 'min_samples_split': 38, 'min_samples_leaf': 23, 'min_weight_fraction_leaf': 0.3969393604796921, 'min_impurity_decrease': 0.1843137060848689, 'max_features': 'log2', 'max_leaf_nodes': 5, 'ccp_alpha': 2.3840858168344416e-05, 'imputer_type': 'simple', 'simple_strategy': 'mean'}. Best is trial 419 with value: 0.3453101317437835.


...


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-27 19:07:36,656] Trial 524 finished with value: 0.3500678414601919 and parameters: {'n_estimators': 926, 'learning_rate': 0.013524919607715694, 'max_depth': 8, 'subsample': 0.3225785032981707, 'min_samples_split': 38, 'min_samples_leaf': 23, 'min_weight_fraction_leaf': 0.3993132038335969, 'min_impurity_decrease': 0.7665652894438078, 'max_features': 'log2', 'max_leaf_nodes': 5, 'ccp_alpha': 2.6866692096377316e-05, 'imputer_type': 'knn', 'knn_neighbors': 6}. Best is trial 419 with value: 0.3453101317437835.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-27 19:07:36,958] Trial 525 finished with value: 0.3469776626292549 and parameters: {'n_estimators': 928, 'learning_rate': 0.015216876621632814, 'max_depth': 8, 'subsample': 0.3222015895920153, 'min_samples_split': 38, 'min_samples_leaf': 26, 'min_weight_fraction_leaf': 0.39897793029051504, 'min_impurity_decrease': 0.18986127282933807, 'max_features': 'log2', 'max_leaf_nodes': 6, 'ccp_alpha': 2.4935107498728117e-05, 'imputer_type': 'knn', 'knn_neighbors': 3}. Best is trial 419 with value: 0.3453101317437835.
[I 2026-04-27 19:07:37,031] Trial 526 finished with value: 0.34788895664095215 and parameters: {'n_estimators': 929, 'learning_rate': 0.013306821187720438, 'max_depth': 8, 'subsample': 0.32427732886145544, 'min_samples_split': 37, 'min_samples_leaf': 26, 'min_weight_fraction_leaf': 0.39689369641743705, 'min_impurity_decrease': 0.13410763776603649, 'max_features': 'log2', 'max_leaf_nodes': 5, 'ccp_alpha': 2.5109331187079422e-05, 'imputer_type': 'knn', 'knn_neighbors': 3}. 


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST RMSE: 0.3453
BEST PARAMETERS:
best_params = {
    "n_estimators": 971,
    "learning_rate": 0.01388668367558402,
    "max_depth": 8,
    "subsample": 0.30872672854426325,
    "min_samples_split": 43,
    "min_samples_leaf": 28,
    "min_weight_fraction_leaf": 0.3945207700562416,
    "min_impurity_decrease": 0.16016521565758096,
    "max_features": "sqrt",
    "max_leaf_nodes": 6,
    "ccp_alpha": 3.910485014559049e-05,
    "imputer_type": "simple",
    "simple_strategy": "mean",
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.3785
  min_impurity_decrease: 0.2137
  min_samples_leaf    : 0.1063
  min_samples_split   : 0.0784
  max_leaf_nodes      : 0.0755
  n_estimators        : 0.0477
  subsample           : 0.0391
  min_weight_fraction_leaf: 0.0271
  imputer_type        : 0.0169
  max_features        : 0.0111
  max_depth           : 0.005

In [7]:
# Check overfit on holdout set with best parameters from tuning

best_params = study_gbm.best_params.copy()
best_params["random_state"] = 42
best_params["verbose"] = 0

imputer_type = best_params.pop("imputer_type")

if imputer_type == "simple":
    strategy = best_params.pop("simple_strategy")
    best_imputer = SimpleImputer(strategy=strategy)
    print(f"SimpleImputer strategy: {strategy}")
elif imputer_type == "knn":
    n_neighbors = best_params.pop("knn_neighbors")
    best_imputer = KNNImputer(n_neighbors=n_neighbors)
    print(f"KNNImputer n_neighbors: {n_neighbors}")
else:
    best_imputer = IterativeImputer(max_iter=10, random_state=42)
    print("IterativeImputer selected")

best_params.pop("simple_strategy", None)
best_params.pop("knn_neighbors", None)
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    best_imputer,
    GradientBoostingRegressor(**best_params)
)

final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print(f"Optuna Val RMSE: {study_gbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")

SimpleImputer strategy: mean
BEST PARAMS: {'n_estimators': 971, 'learning_rate': 0.01388668367558402, 'max_depth': 8, 'subsample': 0.30872672854426325, 'min_samples_split': 43, 'min_samples_leaf': 28, 'min_weight_fraction_leaf': 0.3945207700562416, 'min_impurity_decrease': 0.16016521565758096, 'max_features': 'sqrt', 'max_leaf_nodes': 6, 'ccp_alpha': 3.910485014559049e-05, 'random_state': 42, 'verbose': 0}
Optuna Val RMSE: 0.3453
Holdout Test RMSE: 0.2962


### 10k

In [5]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Parameter tuning settings

timeout_seconds = 60*60*1
no_improvement_trials = 100

# Split the 10k dataset to evaluate on holdout after tuning

X_tuning = train_10k_X.head(8000)
y_tuning = train_10k_y_reg.head(8000)
X_val_holdout = train_10k_X.tail(2000)
y_val_holdout = train_10k_y_reg.tail(2000)

def objective_gbm(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth
        "subsample": trial.suggest_float("subsample", 0.3, 0.8), # Fraction of samples used for training each tree
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 50), # Min samples required to split a node
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 15, 100), # Min samples required in a leaf node
        "min_weight_fraction_leaf": trial.suggest_float("min_weight_fraction_leaf", 0, 0.5), # Min weight fraction required in a leaf node
        "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0, 1.0), # Min impurity decrease required to make a split
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 3, 31), # Max leaf nodes in the tree
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-5, 0.1, log=True), # Complexity parameter for pruning
        "random_state": 42, # Fixed random state for reproducibility
        "verbose": 0, # Set to 0 to disable verbose output during training
    }

    imputer_type = trial.suggest_categorical("imputer_type", ["simple", "knn", "iterative"])
    
    if imputer_type == "simple":
        strategy = trial.suggest_categorical("simple_strategy", ["mean", "median", "most_frequent"])
        imputer = SimpleImputer(strategy=strategy)
        
    elif imputer_type == "knn": # K Nearest Neighbors Imputer
        n_neighbors = trial.suggest_int("knn_neighbors", 3, 15)
        imputer = KNNImputer(n_neighbors=n_neighbors)
        
    else: # iterative
        imputer = IterativeImputer(max_iter=10, random_state=42)

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        imputer,
        GradientBoostingRegressor(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_gbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_gbm.optimize(objective_gbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_gbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_gbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_gbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_gbm)
fig1.show()
    
fig2 = plot_param_importances(study_gbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_gbm,
    params=[
        "n_estimators", "learning_rate", "max_depth", "subsample", 
        "min_samples_split", "min_samples_leaf", "min_weight_fraction_leaf",
        "min_impurity_decrease", "max_features", "max_leaf_nodes",
        "ccp_alpha", "imputer_type"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_10k_history.html")
fig2.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_10k_importance.html")
fig3.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_10k_parallel.html")


[I 2026-04-27 20:07:52,807] A new study created in memory with name: no-name-dde1cbfd-6cfb-4589-850f-afc50cf39134
[I 2026-04-27 20:08:13,733] Trial 3 finished with value: 0.28799737257702607 and parameters: {'n_estimators': 454, 'learning_rate': 0.11491242512726084, 'max_depth': 7, 'subsample': 0.6995723090557557, 'min_samples_split': 29, 'min_samples_leaf': 87, 'min_weight_fraction_leaf': 0.1251051924579083, 'min_impurity_decrease': 0.7256421928878715, 'max_features': 'log2', 'max_leaf_nodes': 4, 'ccp_alpha': 0.00021056325396686678, 'imputer_type': 'simple', 'simple_strategy': 'most_frequent'}. Best is trial 3 with value: 0.28799737257702607.
[I 2026-04-27 20:08:21,053] Trial 5 finished with value: 0.29086685520206756 and parameters: {'n_estimators': 458, 'learning_rate': 0.011244648353112912, 'max_depth': 2, 'subsample': 0.7573840974138557, 'min_samples_split': 28, 'min_samples_leaf': 46, 'min_weight_fraction_leaf': 0.021113215472796543, 'min_impurity_decrease': 0.4106579864137372, '


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-27 20:46:35,974] Trial 166 finished with value: 0.28800359683080873 and parameters: {'n_estimators': 492, 'learning_rate': 0.05897328936588128, 'max_depth': 4, 'subsample': 0.311322891147238, 'min_samples_split': 17, 'min_samples_leaf': 25, 'min_weight_fraction_leaf': 0.15501910998201007, 'min_impurity_decrease': 0.5732811294459104, 'max_features': 'log2', 'max_leaf_nodes': 5, 'ccp_alpha': 2.760559130533814e-05, 'imputer_type': 'simple', 'simple_strategy': 'mean'}. Best is trial 66 with value: 0.28782989539236986.
[I 2026-04-27 20:46:35,976] Trial 163 finished with value: 0.28871186415892724 and parameters: {'n_estimators': 593, 'learning_rate': 0.03634698004235691, 'max_depth': 3, 'subsample': 0.691921380788604, 'min_samples_split': 26, 'min_samples_leaf': 34, 'min_weight_fraction_leaf': 0.17391013881812278, 'min_impurity_decrease': 0.7370600437362702, 'max_features': 'log2', 'max_leaf_nodes': 7, 'ccp_alpha': 0.000397118062057068, 'imputer_type': 'simple', 'simple_strategy'


[Early Stop] No improvement in 100 trials. Stopping optimization.

[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-27 20:46:36,205] Trial 165 finished with value: 0.2884395037981806 and parameters: {'n_estimators': 580, 'learning_rate': 0.06315350026979097, 'max_depth': 4, 'subsample': 0.3914320619241274, 'min_samples_split': 26, 'min_samples_leaf': 38, 'min_weight_fraction_leaf': 0.3553740918730245, 'min_impurity_decrease': 0.42349072271889787, 'max_features': 'log2', 'max_leaf_nodes': 7, 'ccp_alpha': 0.00044575152015241683, 'imputer_type': 'simple', 'simple_strategy': 'mean'}. Best is trial 66 with value: 0.28782989539236986.
[I 2026-04-27 20:46:39,304] Trial 170 finished with value: 0.288327771801397 and parameters: {'n_estimators': 656, 'learning_rate': 0.06532908684232074, 'max_depth': 4, 'subsample': 0.6661299555998844, 'min_samples_split': 7, 'min_samples_leaf': 15, 'min_weight_fraction_leaf': 0.3385743149827103, 'min_impurity_decrease': 0.774553018066863, 'max_features': 'log2', 'max_leaf_nodes': 21, 'ccp_alpha': 9.565874843049087e-05, 'imputer_type': 'simple', 'simple_strategy':


[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-27 20:46:40,529] Trial 169 finished with value: 0.28794926271486526 and parameters: {'n_estimators': 933, 'learning_rate': 0.05908951254900086, 'max_depth': 4, 'subsample': 0.6412149417970942, 'min_samples_split': 17, 'min_samples_leaf': 16, 'min_weight_fraction_leaf': 0.1225647655850302, 'min_impurity_decrease': 0.7706008341791215, 'max_features': 'log2', 'max_leaf_nodes': 7, 'ccp_alpha': 9.477336994129637e-05, 'imputer_type': 'simple', 'simple_strategy': 'mean'}. Best is trial 66 with value: 0.28782989539236986.



[Early Stop] No improvement in 100 trials. Stopping optimization.


[I 2026-04-27 20:46:42,751] Trial 171 finished with value: 0.2896141990072339 and parameters: {'n_estimators': 928, 'learning_rate': 0.037008784208541444, 'max_depth': 3, 'subsample': 0.6969015751431231, 'min_samples_split': 24, 'min_samples_leaf': 34, 'min_weight_fraction_leaf': 0.12376637380731477, 'min_impurity_decrease': 0.7343499681986179, 'max_features': 'log2', 'max_leaf_nodes': 30, 'ccp_alpha': 0.0008469328125853573, 'imputer_type': 'simple', 'simple_strategy': 'mean'}. Best is trial 66 with value: 0.28782989539236986.



[Early Stop] No improvement in 100 trials. Stopping optimization.

BEST RMSE: 0.2878
BEST PARAMETERS:
best_params = {
    "n_estimators": 938,
    "learning_rate": 0.044224154343992905,
    "max_depth": 3,
    "subsample": 0.4440702904540561,
    "min_samples_split": 16,
    "min_samples_leaf": 40,
    "min_weight_fraction_leaf": 0.07834199132955552,
    "min_impurity_decrease": 0.7616676770612814,
    "max_features": "log2",
    "max_leaf_nodes": 22,
    "ccp_alpha": 1.5744809085067783e-05,
    "imputer_type": "iterative",
}

--- PARAMETER IMPORTANCE ---
  ccp_alpha           : 0.4715
  min_impurity_decrease: 0.2709
  n_estimators        : 0.0764
  learning_rate       : 0.0310
  max_features        : 0.0307
  min_weight_fraction_leaf: 0.0254
  subsample           : 0.0223
  max_leaf_nodes      : 0.0215
  imputer_type        : 0.0163
  min_samples_leaf    : 0.0158
  min_samples_split   : 0.0154
  max_depth           : 0.0028


In [6]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_gbm.best_params.copy()

# Extract imputer settings safely before applying scaling trick
imputer_type = best_params.pop("imputer_type")

if imputer_type == "simple":
    strategy = best_params.pop("simple_strategy")
    best_imputer = SimpleImputer(strategy=strategy)
    print(f"SimpleImputer strategy: {strategy}")
elif imputer_type == "knn":
    n_neighbors = best_params.pop("knn_neighbors")
    best_imputer = KNNImputer(n_neighbors=n_neighbors)
    print(f"KNNImputer n_neighbors: {n_neighbors}")
else:
    best_imputer = IterativeImputer(max_iter=10, random_state=42)
    print("IterativeImputer selected")

# Clean up parameter dictionary
best_params.pop("simple_strategy", None)
best_params.pop("knn_neighbors", None)

# Scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

best_params["n_estimators"] = original_trees * 2
best_params["learning_rate"] = original_lr / 2

print(f"\n[Scaling Trick Applied] SKLEARN GBM: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    best_imputer,
    GradientBoostingRegressor(**best_params)
)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Val RMSE:   {study_gbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")
print("="*40)

IterativeImputer selected

[Scaling Trick Applied] SKLEARN GBM: Trees 938 -> 1876, LR 0.0442 -> 0.0221
BEST PARAMS: {'n_estimators': 1876, 'learning_rate': 0.022112077171996453, 'max_depth': 3, 'subsample': 0.4440702904540561, 'min_samples_split': 16, 'min_samples_leaf': 40, 'min_weight_fraction_leaf': 0.07834199132955552, 'min_impurity_decrease': 0.7616676770612814, 'max_features': 'log2', 'max_leaf_nodes': 22, 'ccp_alpha': 1.5744809085067783e-05, 'random_state': 42, 'verbose': 0}

Optuna Val RMSE:   0.2878
Holdout Test RMSE: 0.3208


### 100k

In [5]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Parameter tuning settings

timeout_seconds = 60*60*8
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning

X_tuning = train_100k_X.head(80000)
y_tuning = train_100k_y_reg.head(80000)
X_val_holdout = train_100k_X.tail(20000)
y_val_holdout = train_100k_y_reg.tail(20000)

def objective_gbm(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 20), # Max tree depth
        "subsample": trial.suggest_float("subsample", 0.1, 1.0), # Fraction of samples used for training each tree
        "min_samples_split": trial.suggest_int("min_samples_split", 10, 500), # Min samples required to split a node
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 500), # Min samples required in a leaf node
        "min_weight_fraction_leaf": trial.suggest_float("min_weight_fraction_leaf", 0, 0.5), # Min weight fraction required in a leaf node
        "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0, 1.0), # Min impurity decrease required to make a split
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 5, 512), # Max leaf nodes in the tree
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-5, 0.1, log=True), # Complexity parameter for pruning
        "random_state": 42, # Fixed random state for reproducibility
        "verbose": 0, # Set to 0 to disable verbose output during training
    }

    imputer_type = trial.suggest_categorical("imputer_type", ["simple", "knn", "iterative"])
    
    if imputer_type == "simple":
        strategy = trial.suggest_categorical("simple_strategy", ["mean", "median", "most_frequent"])
        imputer = SimpleImputer(strategy=strategy)
        
    elif imputer_type == "knn": # K Nearest Neighbors Imputer
        n_neighbors = trial.suggest_int("knn_neighbors", 3, 50)
        imputer = KNNImputer(n_neighbors=n_neighbors)
        
    else: # iterative
        imputer = IterativeImputer(max_iter=10, random_state=42)

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        imputer,
        GradientBoostingRegressor(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_gbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_gbm.optimize(objective_gbm, timeout=timeout_seconds, n_jobs=1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_gbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_gbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_gbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_gbm)
fig1.show()
    
fig2 = plot_param_importances(study_gbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_gbm,
    params=[
        "n_estimators", "learning_rate", "max_depth", "subsample", 
        "min_samples_split", "min_samples_leaf", "min_weight_fraction_leaf",
        "min_impurity_decrease", "max_features", "max_leaf_nodes",
        "ccp_alpha", "imputer_type"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_100k_history.html")
fig2.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_100k_importance.html")
fig3.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_100k_parallel.html")


[I 2026-04-29 09:14:00,496] A new study created in memory with name: no-name-2334a3a0-72ca-4c50-97be-50ea9ee6c4a5
[I 2026-04-29 09:14:49,107] Trial 0 finished with value: 0.310732807111893 and parameters: {'n_estimators': 437, 'learning_rate': 0.1667521176194013, 'max_depth': 15, 'subsample': 0.6387926357773329, 'min_samples_split': 86, 'min_samples_leaf': 86, 'min_weight_fraction_leaf': 0.02904180608409973, 'min_impurity_decrease': 0.8661761457749352, 'max_features': 'log2', 'max_leaf_nodes': 497, 'ccp_alpha': 0.021368329072358756, 'imputer_type': 'simple', 'simple_strategy': 'median'}. Best is trial 0 with value: 0.310732807111893.
[I 2026-04-29 09:20:07,395] Trial 1 finished with value: 0.3057824584424259 and parameters: {'n_estimators': 362, 'learning_rate': 0.04777437867054351, 'max_depth': 4, 'subsample': 0.3629301836816964, 'min_samples_split': 189, 'min_samples_leaf': 233, 'min_weight_fraction_leaf': 0.3925879806965068, 'min_impurity_decrease': 0.19967378215835974, 'max_feature


BEST RMSE: 0.3020
BEST PARAMETERS:
best_params = {
    "n_estimators": 602,
    "learning_rate": 0.04590009050693113,
    "max_depth": 4,
    "subsample": 0.3702702077447556,
    "min_samples_split": 343,
    "min_samples_leaf": 242,
    "min_weight_fraction_leaf": 0.020346710861258076,
    "min_impurity_decrease": 0.8240878364564256,
    "max_features": None,
    "max_leaf_nodes": 320,
    "ccp_alpha": 3.276614763080726e-05,
    "imputer_type": "iterative",
}

--- PARAMETER IMPORTANCE ---
  ccp_alpha           : 0.7323
  min_impurity_decrease: 0.0893
  min_weight_fraction_leaf: 0.0526
  subsample           : 0.0313
  max_leaf_nodes      : 0.0302
  max_depth           : 0.0262
  min_samples_leaf    : 0.0237
  n_estimators        : 0.0142
  learning_rate       : 0.0001
  min_samples_split   : 0.0000
  imputer_type        : 0.0000
  max_features        : 0.0000


In [6]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_gbm.best_params.copy()

# Extract imputer settings safely before applying scaling trick
imputer_type = best_params.pop("imputer_type")

if imputer_type == "simple":
    strategy = best_params.pop("simple_strategy")
    best_imputer = SimpleImputer(strategy=strategy)
    print(f"SimpleImputer strategy: {strategy}")
elif imputer_type == "knn":
    n_neighbors = best_params.pop("knn_neighbors")
    best_imputer = KNNImputer(n_neighbors=n_neighbors)
    print(f"KNNImputer n_neighbors: {n_neighbors}")
else:
    best_imputer = IterativeImputer(max_iter=10, random_state=42)
    print("IterativeImputer selected")
    
# Clean up parameter dictionary
best_params.pop("simple_strategy", None)
best_params.pop("knn_neighbors", None)

# Scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

best_params["n_estimators"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10

print(f"\n[Scaling Trick Applied] SKLEARN GBM: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    best_imputer,
    GradientBoostingRegressor(**best_params)
)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Val RMSE:   {study_gbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")
print("="*40)

IterativeImputer selected

[Scaling Trick Applied] SKLEARN GBM: Trees 602 -> 6020, LR 0.0459 -> 0.0046
BEST PARAMS: {'n_estimators': 6020, 'learning_rate': 0.004590009050693113, 'max_depth': 4, 'subsample': 0.3702702077447556, 'min_samples_split': 343, 'min_samples_leaf': 242, 'min_weight_fraction_leaf': 0.020346710861258076, 'min_impurity_decrease': 0.8240878364564256, 'max_features': None, 'max_leaf_nodes': 320, 'ccp_alpha': 3.276614763080726e-05, 'random_state': 42, 'verbose': 0}

Optuna Val RMSE:   0.3020
Holdout Test RMSE: 0.3008


### Whole training data set

In [ ]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Parameter tuning settings

timeout_seconds = 60*60*4
no_improvement_trials = 100

# Split the dataset to evaluate on holdout after tuning

split_index = int(len(train_full_X) * 0.8)

X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_cat[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_cat[split_index:]

def objective_gbm(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 20), # Max tree depth
        "subsample": trial.suggest_float("subsample", 0.1, 1.0), # Fraction of samples used for training each tree
        "min_samples_split": trial.suggest_int("min_samples_split", 10, 500), # Min samples required to split a node
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 500), # Min samples required in a leaf node
        "min_weight_fraction_leaf": trial.suggest_float("min_weight_fraction_leaf", 0, 0.5), # Min weight fraction required in a leaf node
        "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0, 1.0), # Min impurity decrease required to make a split
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 5, 512), # Max leaf nodes in the tree
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-5, 0.1, log=True), # Complexity parameter for pruning
        "random_state": 42, # Fixed random state for reproducibility
        "verbose": 0, # Set to 0 to disable verbose output during training
    }

    imputer_type = trial.suggest_categorical("imputer_type", ["simple", "knn", "iterative"])
    
    if imputer_type == "simple":
        strategy = trial.suggest_categorical("simple_strategy", ["mean", "median", "most_frequent"])
        imputer = SimpleImputer(strategy=strategy)
        
    elif imputer_type == "knn": # K Nearest Neighbors Imputer
        n_neighbors = trial.suggest_int("knn_neighbors", 3, 50)
        imputer = KNNImputer(n_neighbors=n_neighbors)
        
    else: # iterative
        imputer = IterativeImputer(max_iter=10, random_state=42)

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        imputer,
        GradientBoostingRegressor(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_gbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_gbm.optimize(objective_gbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_gbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_gbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_gbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_gbm)
fig1.show()
    
fig2 = plot_param_importances(study_gbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_gbm,
    params=[
        "n_estimators", "learning_rate", "max_depth", "subsample", 
        "min_samples_split", "min_samples_leaf", "min_weight_fraction_leaf",
        "min_impurity_decrease", "max_features", "max_leaf_nodes",
        "ccp_alpha", "imputer_type"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_full_history.html")
fig2.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_full_importance.html")
fig3.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_full_parallel.html")


[I 2026-04-22 00:55:36,440] A new study created in memory with name: no-name-e6e86f3f-4af6-4c6c-bb30-bebed2ab97f3
[I 2026-04-22 00:56:27,391] Trial 3 finished with value: 0.7509030339384353 and parameters: {'boosting_type': 'goss', 'num_leaves': 377, 'max_depth': 2, 'learning_rate': 0.1903428179678173, 'scale_pos_weight': 6.082125486473243, 'min_split_gain': 26.39142477642419, 'min_child_weight': 4.8185524092618045e-05, 'min_child_samples': 290, 'colsample_bytree': 0.5066837006445346, 'reg_alpha': 0.008590451603378712, 'reg_lambda': 0.0006321751034820287, 'colsample_bynode': 0.6989172440719156, 'min_data_per_group': 976, 'max_cat_threshold': 632, 'cat_l2': 4.676533112163103e-06, 'cat_smooth': 34.70861426629406, 'max_cat_to_onehot': 37, 'max_bin': 433, 'top_rate': 0.7237115739628408, 'other_rate': 0.1719406111579936, 'n_estimators': 6050}. Best is trial 3 with value: 0.7509030339384353.
[I 2026-04-22 00:56:59,196] Trial 6 finished with value: 0.7442250170611076 and parameters: {'boostin


BEST AUC: 0.7566
BEST PARAMETERS:
best_params = {
    "boosting_type": "goss",
    "num_leaves": 298,
    "max_depth": 20,
    "learning_rate": 0.002488325776616227,
    "scale_pos_weight": 5.888271675991455,
    "min_split_gain": 1.6197064284930547,
    "min_child_weight": 0.0002644551677947709,
    "min_child_samples": 371,
    "colsample_bytree": 0.5316730321670442,
    "reg_alpha": 0.000426119399156468,
    "reg_lambda": 10.280323813334233,
    "colsample_bynode": 0.5281296236914715,
    "min_data_per_group": 996,
    "max_cat_threshold": 678,
    "cat_l2": 1.640465366517915e-08,
    "cat_smooth": 22.561055503199018,
    "max_cat_to_onehot": 45,
    "max_bin": 454,
    "top_rate": 0.18095067523535055,
    "other_rate": 0.31987875938170135,
    "n_estimators": 6725,
}

--- PARAMETER IMPORTANCE ---
  max_depth           : 0.2899
  min_data_per_group  : 0.1874
  max_bin             : 0.1858
  boosting_type       : 0.0868
  learning_rate       : 0.0573
  min_child_samples   : 0.0480
 

In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_gbm.best_params.copy()

# Extract imputer settings safely before applying scaling trick
imputer_type = best_params.pop("imputer_type")

if imputer_type == "simple":
    strategy = best_params.pop("simple_strategy")
    best_imputer = SimpleImputer(strategy=strategy)
    print(f"SimpleImputer strategy: {strategy}")
elif imputer_type == "knn":
    n_neighbors = best_params.pop("knn_neighbors")
    best_imputer = KNNImputer(n_neighbors=n_neighbors)
    print(f"KNNImputer n_neighbors: {n_neighbors}")
else:
    best_imputer = IterativeImputer(max_iter=10, random_state=42)
    print("IterativeImputer selected")

# Clean up parameter dictionary
best_params.pop("simple_strategy", None)
best_params.pop("knn_neighbors", None)

# Scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

best_params["n_estimators"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10

print(f"\n[Scaling Trick Applied] SKLEARN GBM: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    best_imputer,
    GradientBoostingRegressor(**best_params)
)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Val RMSE:   {study_gbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")
print("="*40)

Optuna Val AUC: 0.7566
Holdout Test AUC: 0.7298
